## Part 4 Vector Databases: Embeddings & Similarity

In [ ]:
# Install required libraries (run once in colab)
!pip install sentence-transformers seaborn matplotlib -q

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

print('All libraries imported successfully')

## Step 1 — Define 10 senences across 3 topics. 

In [ ]:
# 10 sentences: 4 Cricket, 3 Cooking, 3 Cybersecurity
sentences = [
    " The batsman hit a six off the last ball to win the match.",         #0
    "The bowler delivered a yorker that clean bowled the opener.",        #1
    "India won the Test Series after a thrilling final Innings.",         #2
    "The fielder took a stunning catch near the boundary rope.",          #3

    # Cooking (3 sentences)
    "Sauté the onions in olive oil until they turn golden brown.",        #4
    "Add a pinch of salt and let the dough rest for thirty minutes. ",    #5
    "Slow cooking the curry on low heat enhances the depth of flavor.",   #6

    # Cybersecurity (3 sentences)
    "The hacker exploited a SQL injection vulnerability in the login form.", # 7
    "Two-factor authentication significantly reduces the risk of account breaches.", # 8
    "Encrypting sensitive data at rest protects it from unauthorised access."  # 9
]

labels = [
    "Cricket-1", "Cricket-2", "Cricket-3", "Cricket-4",
    "Cooking-1", "Cooking-2", "Cooking-3",
    "Cyber-1",   "Cyber-2",   "Cyber-3"
]

print(f'Total sentences: {len(sentences)}')
for i, s in enumerate(sentences):
    print(f'  [{labels[i]}] {s}')

## Step 2 — Generate Embeddings using all-MiniLM-L6-v2

In [ ]:
# Load the sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for all 10 sentences
embeddings = model.encode(sentences)

print(f'Embedding shape: {embeddings.shape}')
print(f'Each sentence is represented as a {embeddings.shape[1]}-dimensional vector')

## Step 3 — Compute 10×10 Cosine Similarity Matrix

In [ ]:
# Compute cosine similarity between all pairs of sentences
similarity_matrix = cosine_similarity(embeddings)

print('Cosine Similarity Matrix (10x10):')
print(np.round(similarity_matrix, 2))

## Step 4 — Display Heatmap of Cosine Similarity Matrix

In [ ]:
plt.figure(figsize=(12, 9))

sns.heatmap(
    similarity_matrix,
    xticklabels=labels,
    yticklabels=labels,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    vmin=0,
    vmax=1,
    linewidths=0.5,
    linecolor='white'
)

plt.title('10×10 Cosine Similarity Matrix\n(Cricket | Cooking | Cybersecurity)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

print('Observation: Sentences within the same topic cluster show higher similarity scores (warmer colours).')
print('Cross-topic pairs show lower similarity (cooler colours), confirming semantic separation.')

## Step 5 — Query: Find Top 2 Most Similar Sentences

In [ ]:
# Query sentence
query = "The bowler took three wickets in one over"

# Generate embedding for the query
query_embedding = model.encode([query])

# Compute cosine similarity between query and all 10 sentences
query_similarities = cosine_similarity(query_embedding, embeddings)[0]

# Get top 2 most similar sentences (sorted descending)
top2_indices = np.argsort(query_similarities)[::-1][:2]

print(f'Query: "{query}"')
print()
print('Top 2 Most Similar Sentences:')
print('-' * 60)
for rank, idx in enumerate(top2_indices, start=1):
    print(f'Rank {rank}: [{labels[idx]}]')
    print(f'  Sentence  : {sentences[idx]}')
    print(f'  Similarity: {query_similarities[idx]:.4f}')
    print()